# Event-by-event check against OpenMC `collision_track`

Runs the ²⁵²Cf – trans-stilbene case with **mainline OpenMC (≥ 0.15.3, tested against the 0.16.0 API)**, writing in the same run
* `tracks.h5` → reconstruction with `NeutronTrackAnalyzer`;
* `collision_track.h5` → true reaction (MT), target nuclide and energy loss for every collision (Saliba et al., NSE 2026).

The two descriptions of the same collisions are matched by history and position. Output:
1. collision counts and lost energy, true reaction vs reconstruction outcome;
2. identification accuracy for elastic scattering on H and C;
3. two PoliMi collision files (reconstructed and true elastic events) to compare with pyMPPost.

Use an environment with **mainline** OpenMC: `collision_track` is not available in the photonuclear fork.

In [1]:
import os, sys, shutil
import numpy as np
import pandas as pd
import openmc

sys.path.append(os.path.abspath("/Users/loren/WORKSPACE/PHOTON/verification-and-validation/3-coupling-OpenMC-MPPost/src"))
from TrackAnalyzer import NeutronTrackAnalyzer
import collision_track_check as cc

print("OpenMC", openmc.__version__)

OpenMC 0.15.3


In [2]:
# ================================ CONFIGURATION ================================
BASE_DIR  = "/Users/loren/WORKSPACE/PHOTON/verification-and-validation/3-coupling-OpenMC-MPPost"
SAVE_DIR  = os.path.join(BASE_DIR, "results", "collision_track_check", "trans-stilbene_252Cf")
XS_FILE   = "/Users/loren/WORKSPACE/xs/OpenMC/endfb-vii.1-hdf5/cross_sections.xml"   # same library as the paper runs

N_BATCHES, N_PARTICLES = 25, 4000      # 1e5 histories (~125 MB of tracks)
DETECTOR_MAT_ID = 99
RUN_OPENMC = True                      # False: only analyse existing files

TARGETS = {"H-1":  {"A": 0.99917, "zaid": 1001},
           "C-12": {"A": 11.8969, "zaid": 6000}}
E_THRESHOLD  = 1e1                     # eV, as in the paper runs
MAX_RESIDUAL = 0.05
# collision_track nuclide label -> analyzer label
EXPECTED = {"H-1": "H-1", "C-nat": "C-12"}
# ===============================================================================
os.makedirs(SAVE_DIR, exist_ok=True)
TRACK_PATH = os.path.join(SAVE_DIR, "tracks.h5")
CT_PATH    = os.path.join(SAVE_DIR, "collision_track.h5")

## Model (same as the ²⁵²Cf – trans-stilbene paper case)

In [3]:
stilbene = openmc.Material(name="stilbene", material_id=DETECTOR_MAT_ID)
stilbene.set_density("g/cm3", 1.16)
stilbene.add_nuclide("C0", 0.5385)
stilbene.add_nuclide("H1", 0.4615)

air = openmc.Material(name="air")
air.set_density("g/cm3", 1.225e-14)
air.add_nuclide("N14", 0.78)
air.add_nuclide("O16", 0.21)

materials = openmc.Materials([stilbene, air])
materials.cross_sections = XS_FILE

radius, length = 5.08 / 2, 5.08
cyl, bottom, top = openmc.ZCylinder(r=radius), openmc.ZPlane(z0=0), openmc.ZPlane(z0=length)
outer = openmc.Sphere(r=8, boundary_type="vacuum")
det_cell = openmc.Cell(fill=stilbene, region=-cyl & +bottom & -top, cell_id=20)
air_cell = openmc.Cell(fill=air, region=-outer & ~det_cell.region)
geometry = openmc.Geometry(openmc.Universe(cells=[det_cell, air_cell]))

source = openmc.IndependentSource(
    space=openmc.stats.Point((0.0, 0.0, -1.0)),
    angle=openmc.stats.Monodirectional((0.0, 0.0, 1.0)),
    energy=openmc.stats.Watt(a=1.025e6, b=2.926e-6),
    particle="neutron")

settings = openmc.Settings()
settings.run_mode = "fixed source"
settings.seed = 42
settings.source = source
settings.batches, settings.particles = N_BATCHES, N_PARTICLES
settings.photon_transport = False          # neutron collisions only
settings.max_tracks = N_BATCHES * N_PARTICLES
settings.collision_track = {
    "material_ids": [DETECTOR_MAT_ID],     # no reaction filter: record every MT
    "max_collisions": 2_000_000,           # ~0.2 GB of RAM; must exceed the number of collisions
    "max_collision_track_files": 1,        # -> a single collision_track.h5
}
model = openmc.Model(geometry, materials, settings)

In [4]:
if RUN_OPENMC:
    model.run(tracks=True, cwd=SAVE_DIR)
print(sorted(os.listdir(SAVE_DIR)))

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

## Event-by-event comparison

In [5]:
analyzer = NeutronTrackAnalyzer(TARGETS, Ethreshold=E_THRESHOLD, max_residual=MAX_RESIDUAL)

truth = cc.load_truth(CT_PATH, material_ids=[DETECTOR_MAT_ID])
reco  = cc.load_reconstructed(TRACK_PATH, analyzer, material_ids=[DETECTOR_MAT_ID])
print(f"True collisions in the detector : {len(truth)}")
print(f"Collisions found in the tracks  : {len(reco)}")

m = cc.match_events(truth, reco)
print(m["status"].value_counts())

True collisions in the detector : 221172
Collisions found in the tracks  : 221079
status
matched                221079
truth only                 93
reconstruction only         0
Name: count, dtype: int64


In [6]:
pd.set_option("display.width", 250)
counts, energy, edep = cc.summarize(m)
print("Collisions: true reaction (rows) vs reconstruction outcome (columns)")
display(counts)
print("Energy lost by the neutron [MeV], same layout")
display(energy.round(2))
print(edep)

Collisions: true reaction (rows) vs reconstruction outcome (columns)


outcome,not seen (no direction change),rejected: below threshold,rejected: near-forward,rejected: no compatible target,rejected: residual > tolerance,rejected: up-scattering,written as C-12,written as H-1,All
true_label,,,,,,,,,
"C-nat (n,disappear)",51,0,0,0,0,0,0,0,51
"C-nat (n,elastic)",0,702,43,146,370,339,74811,0,76411
"C-nat (n,n1)",0,0,0,158,214,0,0,21,393
"C-nat (n,n2)",0,0,0,1,0,0,0,1,2
"C-nat (n,n3)",0,0,0,0,1,0,0,0,1
"C-nat (n,nc)",0,0,0,2,3,0,0,2,7
"H-1 (n,disappear)",42,0,0,0,0,0,0,0,42
"H-1 (n,elastic)",0,1833,17,591,3112,1340,12,137360,144265
All,93,2535,60,898,3700,1679,74823,137384,221172


Energy lost by the neutron [MeV], same layout


outcome,not seen (no direction change),rejected: below threshold,rejected: near-forward,rejected: no compatible target,rejected: residual > tolerance,rejected: up-scattering,written as C-12,written as H-1,All
true_label,,,,,,,,,
"C-nat (n,disappear)",0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00
"C-nat (n,elastic)",0.0,0.0,0.0,0.00,0.00,-0.0,12674.67,0.00,12674.68
"C-nat (n,n1)",0.0,0.0,0.0,857.82,1031.14,0.0,0.00,101.72,1990.67
"C-nat (n,n2)",0.0,0.0,0.0,8.75,0.00,0.0,0.00,8.36,17.11
"C-nat (n,n3)",0.0,0.0,0.0,0.00,10.27,0.0,0.00,0.00,10.27
"C-nat (n,nc)",0.0,0.0,0.0,24.83,31.48,0.0,0.00,18.72,75.02
"H-1 (n,disappear)",0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00
"H-1 (n,elastic)",0.0,0.0,0.0,32.47,0.01,-0.0,0.00,57377.61,57410.09
All,0.0,0.0,0.0,923.86,1072.90,-0.0,12674.67,57506.41,72177.84


matched written events       212207.0
max |edep_reco - dE| [eV]         0.0
max relative difference           0.0
max |t_reco - t_true| [s]         0.0
dtype: float64


In [7]:
ident = cc.identification_check(m, EXPECTED)
display(ident.round(3))

,true elastic (seen),correct,wrong nuclide,rejected,correct [%],dE rejected / dE total [%]
nuclide,,,,,,
H-1,144265,137360,12,6893,95.214,0.057
C-nat,76411,74811,0,1600,97.906,0.000


### Misidentified inelastic collisions

Inelastic collisions written as elastic carry dE = E − E′, which includes the excitation energy of the target
(e.g. 4.44 MeV for ¹²C). If labelled H-1, pyMPPost converts dE into proton light: check how many and how energetic.

In [8]:
bad = m[(m["status"] == "matched") & m["written"].fillna(False).astype(bool) & (m["mt"] != 2)]
print(f"Non-elastic collisions written as elastic: {len(bad)}")
display(bad.groupby(["true_label", "reco_label"])["dE"].describe())

Non-elastic collisions written as elastic: 24


,,count,mean,std,min,25%,50%,75%,max
true_label,reco_label,,,,,,,,
"C-nat (n,n1)",H-1,21.0,4.843739e+06,16307.289894,4.819002e+06,4.831733e+06,4.838456e+06,4.856035e+06,4.880020e+06
"C-nat (n,n2)",H-1,1.0,8.363625e+06,NaN,8.363625e+06,8.363625e+06,8.363625e+06,8.363625e+06,8.363625e+06
"C-nat (n,nc)",H-1,2.0,9.357798e+06,793399.900603,8.796780e+06,9.077289e+06,9.357798e+06,9.638307e+06,9.918816e+06


In [9]:
for name, df in (("counts", counts), ("energy_MeV", energy), ("identification", ident)):
    df.to_csv(os.path.join(SAVE_DIR, f"collision_track_check_{name}.csv"))
m.to_pickle(os.path.join(SAVE_DIR, "collision_track_check_matched.pkl"))

## PoliMi collision files for the light-output comparison

Run `pyMPPost_analyzer.ipynb` on both files (same TOML, only `polimi_det_in` changes) and compare the two
light-output histograms with the usual pull analysis. Same histories, so differences come only from the reconstruction
(identification, rejected events) and not from statistics.

In [10]:
n_hist = N_BATCHES * N_PARTICLES
reco_file  = os.path.join(SAVE_DIR, f"collision_file_reconstructed_{n_hist:.0e}.d")
truth_file = os.path.join(SAVE_DIR, f"collision_file_collision_track_elastic_{n_hist:.0e}.d")

writer = NeutronTrackAnalyzer(TARGETS, Ethreshold=E_THRESHOLD, max_residual=MAX_RESIDUAL,
                              polimi_file_path=reco_file)
writer.process_file(TRACK_PATH, target_materials=[DETECTOR_MAT_ID])
writer.summary()

n = cc.write_polimi_from_collision_track(CT_PATH, truth_file, material_ids=[DETECTOR_MAT_ID],
                                         zaid_map={1: 1001, 6: 6000}, mts=(2,), Ethreshold=E_THRESHOLD)
print(f"collision_track elastic events written: {n}")

Particle            : neutron
Tracks analysed     : 100000
Tracks w/o events   : 27748
Written events      : 212207
  H-1               : 137384
  C-12              : 74823
collision_track elastic events written: 212833


In [11]:
w = m[m["written"].fillna(False).astype(bool)]
good = w[(w["mt"] == 2) & (w["reco_label"] == "H-1")]["residual"]
bad  = w[w["mt"] != 2]["residual"]
print(good.quantile([0.5, 0.99, 0.999, 1.0]))
print(np.sort(bad.values))

0.500    0.000185
0.990    0.022598
0.999    0.044196
1.000    0.049881
Name: residual, dtype: float64
[0.00336997 0.00339959 0.00734094 0.00856456 0.00905801 0.01002508
 0.01062469 0.01513361 0.01787229 0.01850261 0.02170609 0.02320917
 0.0238573  0.02638643 0.02645334 0.02851003 0.03051613 0.0307029
 0.03323129 0.03557337 0.03903684 0.03953718 0.0409625  0.04152603]
